In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from tqdm import tqdm
import math

### Device Setup

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)


Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM (GB): 3.951296512


### Load mT5 (Embedding Only)

In [3]:
model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

base_model.to(device)
base_model.eval()

for param in base_model.parameters():
    param.requires_grad = False

embedding_layer = base_model.get_input_embeddings()
hidden_size = base_model.config.d_model
vocab_size = base_model.config.vocab_size


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


### Load Dataset (Hindi Only)

In [4]:
dataset = load_dataset("csv", data_files="newspaper_en_hi_translated.csv")

MAX_LEN = 32  # smaller for 4GB GPU

def preprocess(example):
    tokens = tokenizer(
        example["hindi"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )
    return {"input_ids": tokens["input_ids"]}

dataset = dataset.map(preprocess, batched=True)
dataset.set_format(type="torch")

train_loader = DataLoader(
    dataset["train"],
    batch_size=4,
    shuffle=True
)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

### Diffusion Schedule
We define β schedule (linear).

In [5]:
T = 50  # diffusion steps (keep small for GPU)

betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# helper

def get_index_from_list(vals, t, x_shape):
    batch_size = t.shape[0]
    out = vals.gather(-1, t).reshape(batch_size, *((1,) * (len(x_shape) - 1)))
    return out

### Add Noise Function (Forward Process)

In [6]:
def forward_diffusion(x0, t):
    sqrt_alpha_bar = get_index_from_list(torch.sqrt(alpha_bars), t, x0.shape)
    sqrt_one_minus_alpha_bar = get_index_from_list(torch.sqrt(1 - alpha_bars), t, x0.shape)
    
    noise = torch.randn_like(x0)
    xt = sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise
    return xt, noise


### Timestep Embedding

In [7]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear1 = nn.Linear(dim, dim)
        self.linear2 = nn.Linear(dim, dim)
    
    def forward(self, t):
        half_dim = hidden_size // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((torch.sin(emb), torch.cos(emb)), dim=-1)
        emb = self.linear1(emb)
        emb = F.relu(emb)
        emb = self.linear2(emb)
        return emb


### Denoiser Model (Predict Noise)

In [8]:
class DiffusionModel(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        
        self.time_embed = TimeEmbedding(hidden_size)
        
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_size,
                nhead=8,
                dim_feedforward=1024,
                batch_first=True
            ),
            num_layers=2
        )
        
        self.output = nn.Linear(hidden_size, hidden_size)
    
    def forward(self, x, t):
        t_embed = self.time_embed(t)
        t_embed = t_embed.unsqueeze(1)
        x = x + t_embed
        x = self.transformer(x)
        return self.output(x)

model = DiffusionModel(hidden_size).to(device)

### Training Setup

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
EPOCHS = 15

### Training Loop (Real DDPM Objective)

In [10]:
for epoch in range(EPOCHS):
    total_loss = 0
    
    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        
        with torch.no_grad():
            x0 = embedding_layer(input_ids)
        
        t = torch.randint(0, T, (input_ids.size(0),), device=device).long()
        
        xt, noise = forward_diffusion(x0, t)
        
        predicted_noise = model(xt, t)
        
        loss = F.mse_loss(predicted_noise, noise)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")


100%|██████████| 125/125 [00:02<00:00, 42.08it/s]


Epoch 1 | Loss: 1.0349


100%|██████████| 125/125 [00:02<00:00, 60.33it/s]


Epoch 2 | Loss: 1.0162


100%|██████████| 125/125 [00:02<00:00, 58.25it/s]


...

Epoch 13 | Loss: 1.0010


100%|██████████| 125/125 [00:01<00:00, 114.49it/s]


Epoch 14 | Loss: 1.0013


100%|██████████| 125/125 [00:01<00:00, 122.53it/s]

Epoch 15 | Loss: 1.0014


### Reverse Sampling (Generation)

In [11]:
@torch.no_grad()
def sample():
    x = torch.randn(1, MAX_LEN, hidden_size).to(device)
    
    for t in reversed(range(T)):
        t_tensor = torch.full((1,), t, device=device, dtype=torch.long)
        
        predicted_noise = model(x, t_tensor)
        
        beta = betas[t]
        alpha = alphas[t]
        alpha_bar = alpha_bars[t]
        
        x = (1 / torch.sqrt(alpha)) * (
            x - ((1 - alpha) / torch.sqrt(1 - alpha_bar)) * predicted_noise
        )
        
        if t > 0:
            noise = torch.randn_like(x)
            x = x + torch.sqrt(beta) * noise
    
    return x


### Decode Generated Embeddings

In [12]:
@torch.no_grad()
def generate_text():
    embeddings = sample()
    
    logits = torch.matmul(embeddings, embedding_layer.weight.T)
    tokens = torch.argmax(logits, dim=-1)
    
    return tokenizer.decode(tokens[0], skip_special_tokens=True)

print(generate_text())


embry112233お伝えbmRгеолог一把OklaBastanaugura⸢paho日电 கொடு자와4,0.3уудынetməطبعاتасыпmåltid,7,8,9,10,11,12)};149-0-0.÷15¬tryPriceন্ডাの間にക്കുകയും匙onstitucionalSessionId
